# 基于自建搜索引擎的简易 RAG

本示例展示一次完整的 RAG 流程：

```text
用户问题 -> 自建搜索引擎接口 -> Top-K 结果 -> 信息整合接口 -> 上下文 -> 大模型 -> 字符串答案
```

本 notebook 只提供搜索和信息整合接口；同学可以把自己的搜索引擎和处理逻辑接入这条流程。

## 1. 安装依赖

首次运行时取消下面命令的注释。

In [ ]:
# %pip install requests beautifulsoup4 openai

## 2. 填写 API 配置

本示例使用无需 API Key 的 DuckDuckGo HTML 搜索作为演示；公共页面可能受到网络或访问频率限制，同学也可以把 `search()` 替换成自己的搜索引擎。大模型使用 OpenAI 兼容接口。请不要把填写了真实密钥的版本公开上传。

In [ ]:
LLM_API_KEY = 'your_api_key_here'  # 请替换为你的 API Key
LLM_BASE_URL = 'https://api.deepseek.com'
LLM_MODEL = 'deepseek-v4-flash'

## 3. Top-K 搜索接口

`search()` 是统一搜索接口。本示例使用了免费的 DuckDuckGo HTML 搜索进行演示，请同学在使用时把函数体替换为自己实现的搜索引擎。只要返回 `list[SearchResult]` 即可。每条结果建议包含 `url`、`title`、`snippet`（也可以额外提供 `content`）。

In [7]:
from typing import Callable, TypedDict
from urllib.parse import parse_qs, unquote, urlparse

import requests
from bs4 import BeautifulSoup

class SearchResult(TypedDict, total=False):
    url: str
    title: str
    snippet: str
    content: str


def duckduckgo_search(query: str, top_k: int = 5) -> list[SearchResult]:
    """使用 DuckDuckGo 公共 HTML 页面搜索，不需要 API Key。"""
    response = requests.get(
        'https://html.duckduckgo.com/html/',
        params={'q': query},
        headers={'User-Agent': 'Mozilla/5.0 RAG-Course-Demo'},
        timeout=10,
    )
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    results: list[SearchResult] = []
    for node in soup.select('.result'):
        link = node.select_one('.result__a')
        if link is None:
            continue
        href = link.get('href', '')
        query_params = parse_qs(urlparse(href).query)
        href = unquote(query_params.get('uddg', [href])[0])
        snippet_node = node.select_one('.result__snippet')
        results.append({
            'url': href,
            'title': ' '.join(link.get_text(' ', strip=True).split()),
            'snippet': (snippet_node.get_text(' ', strip=True) if snippet_node else ''),
        })
        if len(results) >= top_k:
            break
    return results


def search(query: str, top_k: int = 5) -> list[SearchResult]:
    """统一搜索接口；课堂演示默认使用 DuckDuckGo。"""
    # 同学可以把这里替换为自己的搜索引擎实现。
    return duckduckgo_search(query, top_k=top_k)


In [8]:
# DuckDuckGo 是免费演示选项；运行时需要联网。
test_results = search('中国人民大学高瓴人工智能学院', top_k=5)
test_results

[{'url': 'http://ai.ruc.edu.cn/',
  'title': '首页_中国人民大学高瓴人工智能学院',
  'snippet': '智汇未来，青年引领——人工智能青年科学家论坛成功举办 智启新程，勇走新路——高瓴人工智能学院2026级本科新生报到工作圆满完成 中国人民大学文继荣教授在SIGIR 2026做大会主旨特邀报告 高瓴人工智能学院2026届同学毕业快乐!'},
 {'url': 'http://ai.ruc.edu.cn/overview/intro/index.htm',
  'title': '学院简介_中国人民大学高瓴人工智能学院 - ruc.edu.cn',
  'snippet': '学院简介_中国人民大学高瓴人工智能学院 高瓴人工智能学院负责学校人工智能学科的规划与建设。学院由高瓴资本创始人、中国人民大学校友张磊先生捐资支持建设。学院以打造一所能够影响和塑造未来人工智能时代的世界一流学院为愿景，培养有人类情怀（有温度）的人工智能本硕博人才。 作为 ...'},
 {'url': 'http://www.lianpp.com/ruc/mu_ai//index.htm',
  'title': '首页_中国人民大学高瓴人工智能学院',
  'snippet': '首页_中国人民大学高瓴人工智能学院 高瓴人工智能学院师生获ACL 2026杰出论文奖 07-14 高瓴人工智能学院师生获WWW 2026唯一最佳长文奖 07-14 曾毅参加联合国首届人工智能治理全球对话全体会议并发言 07-14 瓴动夏夜!这一晚，毕业音符汇成璀璨星河 06-23 中国社会科学杂志社与中国人民大学高瓴人工 ...'},
 {'url': 'http://www.lianpp.com/ruc/mu_ai/index.htm',
  'title': '首页_中国人民大学高瓴人工智能学院',
  'snippet': '融入智能浪潮，开启未来征程——高瓴人工智能学院2025级本科新生报到工作圆满完成 高瓴人工智能学院2025届同学毕业快乐! 《大语言模型》：人工智能时代的知识盛宴，大模型中文书籍震撼发售! 新闻 更多'},
 {'url': 'https://baike.baidu.com/item/中国人民大学理工学部高瓴人工智能学院/6494292

## 4. 信息整合：把检索结果变成可用上下文

检索结果通常不能直接交给模型，需要先做信息整合。本节先完整演示最简单的 snippet merge，再保留 full 和 custom 接口供同学扩展：摘要压缩、关键信息抽取、相关性重排、去重、正文截取等。

- `snippet`：调用 `snippet_merge()` 直接使用搜索摘要，速度快，适合作为起点。
- `full`：调用 `full_merge()` 读取网页或本地文档正文，信息更完整，但需要自行处理长度和异常。
- `custom`：调用 `custom_integrator()`，按查询问题进行压缩、抽取、去重或重排。

建议先实现最简单的 `snippet` 版本，再逐步加入正文解析、规则过滤或模型压缩。

In [9]:
def snippet_merge(
    results: list[SearchResult],
    max_chars: int = 12_000,
) -> str:
    """将 Top-K 搜索摘要清洗、去重并组织为上下文。"""
    documents: list[str] = []
    seen_urls: set[str] = set()
    seen_texts: set[str] = set()

    for item in results:
        title = ' '.join(str(item.get('title', '') or '').split())
        url = str(item.get('url', '') or '').strip()
        text = item.get('snippet') or item.get('content') or ''
        text = ' '.join(str(text).split())
        if not text or (url and url in seen_urls) or text in seen_texts:
            continue
        if url:
            seen_urls.add(url)
        seen_texts.add(text)
        documents.append(
            f'[文档{len(documents) + 1}]\n'
            f'标题：{title}\n'
            f'URL：{url}\n'
            f'内容：{text}'
        )

    return '\n\n'.join(documents)[:max_chars]


def full_merge(
    results: list[SearchResult],
    max_chars: int = 12_000,
) -> str:
    """full 整合接口：读取网页或本地文档正文后组织上下文。"""
    # 可在这里实现下载、HTML 清洗、分块、截断和异常处理。
    raise NotImplementedError(
        '请实现 full merge，并返回可放入 Prompt 的上下文字符串'
    )


def custom_integrator(
    results: list[SearchResult],
    query: str = '',
    max_chars: int = 12_000,
) -> str:
    """custom 整合接口：按问题压缩、抽取、去重或重排。"""
    # 可以接入规则、关键词筛选或其他重合方法。
    raise NotImplementedError(
        '请实现 custom integrator，并返回可放入 Prompt 的上下文字符串'
    )


In [10]:
# 信息整合分发器：只负责选择策略，不实现具体处理逻辑。
def integrate_information(
    results: list[SearchResult],
    strategy: str = 'snippet',
    query: str = '',
) -> str:
    """按照 strategy 调用三种整合接口之一。"""
    if strategy == 'snippet':
        return snippet_merge(results)
    if strategy == 'full':
        return full_merge(results)
    if strategy == 'custom':
        return custom_integrator(results, query=query)
    raise ValueError('strategy 必须是 snippet、full 或 custom')

In [11]:
# snippet merge 的本地小例子：不调用网络，也不需要大模型 Key。
demo_results = [
    {
        'url': 'https://example.com/ai',
        'title': '人工智能学院简介',
        'snippet': '学院面向人工智能基础理论、关键技术与应用开展教学和科研。',
    },
    {
        'url': 'https://example.com/ai-en',
        'title': '学院英文名称',
        'snippet': '学院英文名称为 Gaoling School of Artificial Intelligence。',
    },
]
print(snippet_merge(demo_results))

# 三种接口都可以继续拆成多个阶段：
# normalize -> deduplicate -> rerank -> compress/extract -> format_context
# 先完成 snippet_merge，再尝试 full_merge 或 custom_integrator。


[文档1]
标题：人工智能学院简介
URL：https://example.com/ai
内容：学院面向人工智能基础理论、关键技术与应用开展教学和科研。

[文档2]
标题：学院英文名称
URL：https://example.com/ai-en
内容：学院英文名称为 Gaoling School of Artificial Intelligence。


## 5. 调用大模型

检索结果可能包含重复、冗余或与问题不完全相关的内容。信息整合阶段负责把它们处理成模型易于理解的上下文：可以保留摘要，也可以进行正文提取、压缩、抽取、去重或重排。

In [13]:
from openai import OpenAI


def call_model(prompt: str) -> str:
    if LLM_API_KEY == 'your_api_key':
        raise RuntimeError('请先填写 LLM_API_KEY')

    client = OpenAI(
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        timeout=55.0,
        max_retries=0,
    )
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                'role': 'system',
                'content': '你是一个基于检索材料进行事实问答的 RAG 助手。',
            },
            {'role': 'user', 'content': prompt},
        ],
        temperature=0.0,
        max_tokens=512,
        stream=False,
    )
    return (response.choices[0].message.content or '').strip()

print(call_model("介绍一下中国人民大学高瓴人工智能学院"))   

中国人民大学高瓴人工智能学院是中国人民大学下属的二级学院，成立于2019年，由高瓴资本创始人张磊捐资支持建设。学院致力于人工智能领域的前沿研究与应用，重点方向包括机器学习、自然语言处理、计算机视觉、数据挖掘、智能博弈与决策等。学院拥有完整的本硕博培养体系，并与多家企业及研究机构合作，推动人工智能与人文社科交叉融合，强调“有温度的人工智能”理念。院长由文继荣教授担任。


## 6. 完整 RAG 流程

`search_fn` 是预留给同学的搜索引擎接口，默认指向上面的 `search()` 占位函数；运行时请传入自己的实现。

In [14]:
def rag_answer(
    query: str,
    top_k: int = 5,
    strategy: str = 'snippet', # 信息整合的方式
    search_fn: Callable = search, # 搜索引擎
) -> str:
    """完成一次“检索 + 信息整合 + 生成”的简单 RAG。"""
    # 第一步：检索与问题最相关的 Top-K 条结果。
    # TODO 调用自己的搜索引擎
    results = search_fn(query, top_k=top_k)

    # 第二步：整合检索结果，形成模型可以阅读的上下文。
    # TODO 实现自己的RAG信息整合方式
    context = integrate_information(results, strategy=strategy, query=query)
    if not context:
        return '未检索到足够信息'

    # 第三步：把问题和整合后的材料一起放入 Prompt。
    # TODO prompt可按自己的需求设计随意更改。
    prompt = f"""请根据检索材料简洁回答问题；材料不足时明确说明，不要猜测或补充材料之外的事实。
                 问题：{query}
                 检索材料：{context}
                 答案："""
    
    # 第四步：调用大模型，得到最终答案。
    return call_model(prompt)

rag_answer('北京今天的天气怎么样', top_k=5, strategy='snippet', search_fn=search)

'根据检索材料，北京今天天气为**阴**，实况温度约26.9℃，最高29℃/最低21℃，西风1级，相对湿度68%。'

In [20]:
# 课堂展示：RAG 文档结构与模型回答（可直接截图放入 PPT）
# 真实运行时，下面的 context 来自 integrate_information，answer 来自 call_model。
demo_context = """[文档1]
标题：中国人民大学高瓴人工智能学院
URL：https://ai.ruc.edu.cn/
内容：英文名为 Gaoling School of Artificial Intelligence，成立于 2020 年。

[文档2]
标题：学院简介
URL：https://ai.ruc.edu.cn/about
内容：学院开展人工智能基础理论、关键技术与应用研究。"""

demo_answer = '中国人民大学高瓴人工智能学院的英文名称是 Gaoling School of Artificial Intelligence。'

print('【RAG 文档结构】')
print(demo_context)
print('\n【模型回答】')
print(demo_answer)

【RAG 文档结构】
[文档1]
标题：中国人民大学高瓴人工智能学院
URL：https://ai.ruc.edu.cn/
内容：英文名为 Gaoling School of Artificial Intelligence，成立于 2020 年。

[文档2]
标题：学院简介
URL：https://ai.ruc.edu.cn/about
内容：学院开展人工智能基础理论、关键技术与应用研究。

【模型回答】
中国人民大学高瓴人工智能学院的英文名称是 Gaoling School of Artificial Intelligence。


## 7. 与课程评测接口对应

正式代码中的对应关系：

```text
自建搜索引擎           ->  search(query, top_k)
integrate_information ->  snippet / full / custom
rag_answer                    ->  rag_evaluate(query) -> str
```

评测时每道题只能返回一个字符串，并应在 60 秒内完成。

### 整体流程
```python
RAG问答(查询 query):

    # 1. 检索：可以替换成自己的搜索引擎
    检索结果 = 搜索引擎.search(query,top_k)

    # 2. 信息整合：清洗、去重、抽取或压缩
    如果 整合方式 == "snippet":          # 推荐，快速
        context = 清洗并组织所有结果的 Snippet

    否则如果 整合方式 == "full":         # 信息更完整
        context = 提取并组织所有网页正文

    否则如果 整合方式 == "custom":        # 可扩展
        原始材料 = 摘要或网页正文
        关键信息 = 压缩、抽取、去重或重排(原始材料, query)
        context = 组织关键信息

    # 3. 构造提示词
    prompt = ""
    # 4. 调用大模型
    answer = 大模型.generate(prompt)

    # 5. 每个问题只返回一个字符串
    返回 answer
```